In [1]:
import os
# Force CPU backend on Apple Silicon to avoid Metal issues
os.environ['JAX_PLATFORMS'] = 'cpu'

import matplotlib.pyplot as plt

# Disable LaTeX rendering in matplotlib
plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "sans-serif"

from jax import random
from jax import numpy as jnp
from sbijax import plot_loss_profile

from aind_behavior_vrforaging_analysis.sbi_ddm_analysis.simulator import PatchForagingDDM_JAX, create_prior
from aind_behavior_vrforaging_analysis.sbi_ddm_analysis.snle.snle_inference_jax import train_snle, infer_parameters_snle
from aind_behavior_vrforaging_analysis.sbi_ddm_analysis.snle.snle_utils_jax import plot_real_synth_hist

In [ ]:
# --- Setup ---
num_window_sites = 100
n_simulations = 500_000
simulator = PatchForagingDDM_JAX(max_sites_per_window=num_window_sites)

# Get prior bounds for JAX simulator
prior_fn = create_prior()
rng_key = random.PRNGKey(0)

# --- Train SNLE ---
print("\n1. Training SNLE model...")
snle, snle_params, losses, rng_key, y_mean, y_std = train_snle(
    simulator, 
    prior_fn,
    mode='multi', 
    n_simulations=500_000,
    n_iter=1000,                        # Ensure full 1000 iterations possible
    n_early_stopping_patience=50,       # More patience for convergence
    batch_size=100,                     # Can adjust based on memory
    rng_key=rng_key
)

# Disable LaTeX rendering in matplotlib
plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "sans-serif"

_, axes = plt.subplots(figsize=(6, 3))
plot_loss_profile(losses, axes)
plt.show()


1. Training SNLE model...

Initializing SNLE (multi mode)...
Data dimension: 7

Simulating 500000 training samples...
Training data shapes: theta=(500000, 4), x=(500000, 7)
Training SNLE...


  4%|▍         | 42/1000 [01:37<37:39,  2.36s/it]

In [ ]:
losses

In [ ]:
# --- Simulate observed data ---
print("\n2. Simulating observed data...")
true_theta = jnp.array([0.4, 0.3, 0.1, 0.1])
rng_key, subkey = random.split(rng_key)
_, observed_stats = simulator.simulate_one_window(true_theta, subkey)
print(f"   True theta: {true_theta}")
print(f"   Observed stats: {observed_stats}")

In [ ]:
# --- Run inference ---
print("\n3. Testing inference...")
rng_key, subkey = random.split(rng_key)
posterior_samples, diagnostics = infer_parameters_snle(
snle,
snle_params,
observed_stats, 
y_mean, y_std,
num_samples=100_000,
num_warmup=50,
num_chains=2,
rng_key=subkey
)

In [ ]:
# --- Plot posterior distributions ---
param_names = ["drift_rate", "reward_bump", "failure_bump", "noise_std"]
param_labels = [
    "drift_rate: evidence accumulation rate",
    "reward_bump: evidence boost from receiving reward",
    "failure_bump: evidence boost from not receiving reward",
    "noise_std: std of noise in evidence accumulation"
]

fig, axes = plt.subplots(1, 4, figsize=(10, 2))
axes = axes.flatten()

for i in range(4):

    # Compute histogram
    counts, bins, _ =axes[i].hist(posterior_samples[:, i], bins=30, color='dodgerblue', edgecolor=None, alpha=0.7)

    # Posterior mode (bin center with max count)
    mode_index = jnp.argmax(counts)
    posterior_mode = (bins[mode_index] + bins[mode_index + 1]) / 2

    axes[i].axvline(true_theta[i], color='orangered', linestyle='--', label="true value")
    axes[i].axvline(posterior_mode, color='k', linestyle='--', label='MAP estimate')

    axes[i].set_xlabel(param_names[i])
    axes[i].set_ylabel("Frequency")

axes[i].legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

def pairplot(posterior_samples, true_params=None, param_names=None, figsize_per_param=2.5, grid_points=100):
    """
    Lower-triangle corner plot with:
    - 2D filled KDEs (off-diagonal)
    - 1D KDEs (diagonal)
    - Red 'X' for true parameters
    """
    if isinstance(posterior_samples, jnp.ndarray):
        posterior_samples = np.array(posterior_samples)
    
    n_params = posterior_samples.shape[1]
    if param_names is None:
        param_names = [f"param{i}" for i in range(n_params)]
    
    fig, axes = plt.subplots(n_params, n_params, figsize=(figsize_per_param*n_params, figsize_per_param*n_params))
    
    for i in range(n_params):
        for j in range(n_params):
            ax = axes[i, j]
            
            # Only fill lower triangle
            if i < j:
                ax.axis('off')
                continue
            
            # Diagonal: 1D KDE
            if i == j:
                data = posterior_samples[:, i]
                kde = gaussian_kde(data)
                x_grid = np.linspace(data.min(), data.max(), grid_points)
                ax.fill_between(x_grid, kde(x_grid), color="skyblue")
                
                if true_params is not None:
                    ax.axvline(true_params[i], color='red', linestyle='--', lw=1)
            
            # Off-diagonal: 2D KDE
            else:
                x = posterior_samples[:, j]
                y = posterior_samples[:, i]
                xy = np.vstack([x, y])
                kde = gaussian_kde(xy)
                x_grid = np.linspace(x.min(), x.max(), grid_points)
                y_grid = np.linspace(y.min(), y.max(), grid_points)
                X, Y = np.meshgrid(x_grid, y_grid)
                Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
                ax.contourf(X, Y, Z, levels=20, cmap="Blues")
                
                if true_params is not None:
                    ax.scatter(true_params[j], true_params[i], c='red', s=50, marker='X', label='True')
            
            # Only label left and bottom axes
            if i < n_params - 1:
                ax.set_xticklabels([])
            else:
                ax.set_xlabel(param_names[j])
            if j > 0:
                ax.set_yticklabels([])
            else:
                ax.set_ylabel(param_names[i])
    
    # Add a legend in the top-left subplot
    handles = []
    if true_params is not None:
        handles.append(plt.Line2D([0], [0], marker='X', color='w', markerfacecolor='red', markersize=8, label='True'))
    axes[0, 1].legend(handles=handles, loc='upper left')
    
    plt.tight_layout()
    plt.show()

In [ ]:
pairplot(posterior_samples, true_theta, param_names, figsize_per_param=2.0)

In [ ]:
# Generate multiple patches from posterior samples to compare distributions
print("\nGenerating patches from posterior samples for comparison...")

num_window_sites = 500

#--- Simulate 'real' data from simulator---
print("\n2. Simulating observed data...")
real_data = []
for i_site in range(num_window_sites):
    rng_key, subkey = random.split(rng_key)
    _, data = simulator.simulate_one_window(true_theta, subkey)
    real_data.append(data)
real_data = np.array(real_data)

# # 2. Generate "synthetic" data from simulator using posterior-sampled parameters

for i in range(20):
    rng_key, sample_key = random.split(rng_key)
    observable_norm = (real_data[i,:] - y_mean)/y_std
    synthetic_data, _ = snle.simulate_data(sample_key, params = snle_params, observable = observable_norm, n_simulations=num_window_sites)
    synthetic_data = np.array(synthetic_data['y'])

    plot_real_synth_hist(real_data,synthetic_data)

